In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## 1. Cấu hình

In [ ]:
DATA_PATH   = r"D:\TTNT\Dataset\Dataset_VIE.csv"
SVM_PATH    = "svm_fake_model.pkl"
TFIDF_PATH  = "tfidf.pkl"

TEST_SIZE   = 0.3
SEED        = 42

# TF-IDF
MAX_FEATURES = 50_000
MAX_DF       = 0.85   # Bỏ từ xuất hiện trong > 85% văn bản (quá phổ biến, không có nghĩa)
MIN_DF       = 2      # Bỏ từ xuất hiện < 2 lần (quá hiếm, dễ gây overfitting)

# SVM
SVM_C        = 0.5    # Regularization: nhỏ hơn → margin rộng hơn, ít overfit hơn
SVM_CV       = 5      # Số fold dùng để calibrate xác suất

## 2. Load & chuẩn bị dữ liệu

In [ ]:
df = pd.read_csv(DATA_PATH)

# Ghép title + content thành một chuỗi duy nhất làm đầu vào
df["text"] = df["title"].fillna("") + " " + df["content"].fillna("")

X = df["text"]
y = df["is_fake"]

print(f"Dataset size: {len(df)}")
print(y.value_counts())

## 3. Train / Test split · TF-IDF

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=SEED,
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

vectorizer   = TfidfVectorizer(
    max_df=MAX_DF,
    min_df=MIN_DF,
    max_features=MAX_FEATURES,
    sublinear_tf=True,   # Dùng log(tf) thay vì tf thô — giảm ảnh hưởng của từ lặp nhiều
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

## 4. Train SVM

In [ ]:
# CalibratedClassifierCV bọc LinearSVC để có predict_proba (LinearSVC không hỗ trợ native)
svm_model = CalibratedClassifierCV(
    LinearSVC(C=SVM_C, class_weight="balanced", max_iter=2000, random_state=SEED),
    method="sigmoid",
    cv=SVM_CV,
)
svm_model.fit(X_train_tfidf, y_train)
print("✅ Train xong!")

## 5. Đánh giá

In [ ]:
y_pred  = svm_model.predict(X_test_tfidf)
y_proba = svm_model.predict_proba(X_test_tfidf)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

confidence = np.max(y_proba, axis=1)
print(f"Độ tự tin trung bình    : {confidence.mean():.3f}")
print(f"% dự đoán > 80% tự tin : {(confidence > 0.8).mean() * 100:.1f}%")

## 6. Lưu model

In [ ]:
joblib.dump(svm_model,  SVM_PATH)
joblib.dump(vectorizer, TFIDF_PATH)
print(f"✅ Đã lưu: {SVM_PATH}, {TFIDF_PATH}")